# Module 7.1: PEFT and LoRA (Low-Rank Adaptation)

Welcome to Phase 3: Applied LLM Engineering! We know how to build and train Transformers, but there is an elephant in the room: **Hardware**.

If you want to Supervised Fine-Tune (SFT) a 70 Billion parameter model on your own dataset, doing standard backpropagation requires hundreds of gigabytes of VRAM. Nobody has that at home.

In this notebook, we look at answering three questions: **What** is PEFT, **Why** is it needed, and **How** does LoRA perform mathematical black magic to solve the memory crisis?

## 1. WHAT is PEFT?

**Parameter-Efficient Fine-Tuning (PEFT)** is the blanket term for methods that train a massive model without touching the original weights.

### \ud83d\udcdd The Dictionary Analogy
Imagine you own a heavy, 5,000-page Encyclopedia (our 70B parameter Base Model). 
You want to learn some new 2024 internet slang (Fine-Tuning).

**Standard Fine-Tuning**: You rip out every page of the encyclopedia, rewrite the definitions, and re-bind the entire heavy book. This takes massive effort (compute/memory).

**PEFT**: You take a tiny, lightweight stack of sticky notes. You write the new slang on the sticky notes and paste them onto the back cover of the Encyclopedia. When you need a word, you check the sticky notes first, then check the book. The heavy book stays completely untouched (frozen)!

## 2. WHY do we need LoRA?

When you use standard fine-tuning, you load the model weights. But when PyTorch computes gradients (the slopes) and the optimizer keeps its state (AdamW's two momentum buffers), it allocates *new* buffers, each the same size as the weights.

So roughly: if the weights are $10$ GB, the gradients add $10$ GB, and AdamW's two state buffers add about $20$ GB more. That is around **$40$ GB** of memory, before counting activations, just to train.

**LoRA (Low-Rank Adaptation)** is the most popular form of PEFT. It cuts most of this cost because it only allocates gradients and optimizer state for tiny added matrices, not for the massive frozen weights.

### The key insight: the *update* has low rank

Here is the idea that makes LoRA work. The pretrained weight matrix $W$ is full rank, it genuinely uses all its dimensions. But the *change* you need during fine-tuning, $\Delta W$, turns out to have a low **intrinsic rank**: adapting a model to a new task moves the weights in only a few effective directions.

If $\Delta W$ can be well approximated by something low-rank, you don't need a full-size matrix to represent it. You can write $\Delta W \approx B A$, where $A$ and $B$ are skinny matrices with a small inner dimension $r$. That is far fewer numbers to train, while still capturing the adaptation.

## 3. HOW does LoRA work? (The Math)

LoRA works by hijacking the dense linear layers (`nn.Linear`) in the attention blocks.

Instead of updating the master weight matrix $W$, it freezes $W$ and learns the *update* $\Delta W = B A$ as two tiny matrices $A$ and $B$. The new forward pass is:

$$y = W x + (B A)\, x \cdot \text{scaling}$$

Because $A$ and $B$ share a small inner dimension $r$, together they hold far fewer numbers than a full-size $\Delta W$.

> **A note on shapes (so the diagram and the code agree).** A `nn.Linear(in, out)` stores its weight *transposed*, as shape `(out, in)`, because it computes `x @ weight.T`. So conceptually we think of $W$ as `(in -> out)`, but in code `lora_layer.W.weight.shape` is `(out, in)`. In the diagram below we label dimensions in the conceptual `in x out` direction; just remember PyTorch holds them flipped.

### 🚰 The Pipe Analogy (Low-Rank Compression)
Imagine pouring a giant lake of water (the inputs into matrix $A$) through an ultra-thin garden hose (the bottleneck "rank"), then spraying it back out to fill another lake (the outputs of matrix $B$). The rank $r$ is how thin the hose is. A thinner hose uses less memory but lets less information through.

```mermaid
graph TD
    Input[Tokens X] --> W[Frozen Model Weights W<br>Dim: 4096 x 4096<br>Params: 16 Million!]

    Input -.->|LoRA Injection| A[Matrix A<br>Dim: 4096 x 'r']
    A -.->|The 'r' bottleneck<br>e.g. r=8| B[Matrix B<br>Dim: 'r' x 4096]

    W --> Sum{+}
    B -.->|Params: 65 Thousand!| Sum

    Sum --> Out[Output]
```

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        # 1. The Encyclopedia (the original frozen weights).
        # Note: nn.Linear(in, out) stores its weight as shape (out, in) and computes x @ W.T
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.W.weight.requires_grad = False  # FREEZE! No gradient/optimizer memory here.

        # 2. The Sticky Notes (the LoRA A and B matrices).
        # A maps down to `rank`, B maps back up to `out_features`.
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)

        # alpha / rank is a fixed scaling on the LoRA update. A common rule of thumb is
        # alpha ~= 2 * rank, which keeps the update's strength stable as you change rank.
        self.scaling = alpha / rank

        # Zero-initialize B so LoRA outputs exactly 0 at the start of training.
        # The model therefore behaves EXACTLY like the frozen base model until training begins.
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        frozen_out = self.W(x)                       # frozen base path
        lora_out = self.B(self.A(x)) * self.scaling  # tiny low-rank update
        return frozen_out + lora_out


# Let's visualize the parameter savings!
d_model = 4096
lora_layer = LoRALinear(in_features=d_model, out_features=d_model, rank=8)

# Shape sanity check: nn.Linear stores weight transposed as (out, in).
print(f"W.weight.shape (stored as out x in): {tuple(lora_layer.W.weight.shape)}\n")

frozen_params = sum(p.numel() for p in lora_layer.W.parameters())
trainable_params = sum(p.numel() for p in lora_layer.A.parameters()) + sum(p.numel() for p in lora_layer.B.parameters())

print(f"Original Model Params (Frozen):   {frozen_params:,}")
print(f"LoRA Trainable Params  (Rank 8):  {trainable_params:,}")
print(f"\nWe reduced the count of TRAINED parameters by {((frozen_params - trainable_params) / frozen_params) * 100:.2f}%!")

# --- See the effect of alpha (scaling) on the size of the LoRA update ---
print("\n--- Effect of alpha (with rank fixed at 8) ---")
x = torch.randn(1, d_model)
for alpha in [4, 8, 16, 32]:
    layer = LoRALinear(d_model, d_model, rank=8, alpha=alpha)
    # B starts at zero, so force a non-zero B to actually see the update's magnitude.
    nn.init.normal_(layer.B.weight, std=0.02)
    update = (layer.B(layer.A(x)) * layer.scaling)
    print(f"alpha={alpha:>2}  scaling={layer.scaling:>4}  ||LoRA update|| = {update.norm().item():.4f}")
print("-> Larger alpha => louder update. With B zero-initialized, the real run starts at 0 and grows.")

## Summary

By freezing the 16+ million-parameter matrix ($W$) and training only the ~65 thousand-parameter matrices ($A$ and $B$), the optimizer doesn't need to keep gradient and momentum buffers for the bulk of the model.

When you download a "LoRA adapter" from the internet (often a ~100 MB file instead of a 20 GB model), you are downloading exactly these small $A$ and $B$ matrices. You then use them in one of two ways:

- **Keep them separate**: run the extra `B(A(x)) * scaling` path on every forward pass alongside the frozen base, as in the code above. This lets you hot-swap different adapters on the same base model.
- **Merge once**: fold the update into the base weights a single time with `W += (B A) * scaling`. After merging there is no extra path and no inference overhead, but you've baked that one adapter in.

So merging is a deliberate, one-time step, not something that happens dynamically on every forward pass.

### 🏋️ Try it yourself

1. **Verify the merge equals the separate path.** Give `B` some non-zero weights (`nn.init.normal_(lora_layer.B.weight, std=0.02)`). Build the merged weight `W_merged = lora_layer.W.weight + (lora_layer.B.weight @ lora_layer.A.weight) * lora_layer.scaling`, then check that `x @ W_merged.T` matches `lora_layer(x)` with `torch.allclose`. (Watch the transpose: weights are stored as `(out, in)`.)
2. **Rank vs. parameter count.** Loop `rank` over `[1, 4, 8, 16, 64]` and print the trainable parameter count for each. How does it scale with rank?

In [ ]:
# Your code here!
# Hint for task 1:
# nn.init.normal_(lora_layer.B.weight, std=0.02)
# x = torch.randn(2, d_model)
# W_merged = lora_layer.W.weight + (lora_layer.B.weight @ lora_layer.A.weight) * lora_layer.scaling
# print(torch.allclose(x @ W_merged.T, lora_layer(x), atol=1e-5))